# 요인분석(Factor Analysis) 레포트
### AI 빅데이터 20221393 황태하
## 항공사 승객 만족도 조사 데이터를 활용한 서비스 품질 요인 탐색

본 레포트는 사회과학 분야(소비자/고객 만족도 조사) 데이터셋에 요인분석(탐색적 요인분석, EFA)을
적용한 과정과 결과를 정리한 것입니다. 수업 시간에 실습한 요인분석 절차(적합성 검정 - 요인 수
결정 - 요인추출 및 회전 - 요인적재량 해석 - 공통성 확인 - 요인점수 산출)를 동일하게 따르되,
본 과제에서 직접 수집한 데이터셋에 맞게 적용했습니다.

### 분석 절차 개요
1. 데이터셋 소개 및 분석 목적
2. 데이터 불러오기 및 전처리
3. 탐색적 데이터 분석(EDA) - 문항 간 상관관계 확인
4. 요인분석 적합성 검정 (Bartlett 구형성 검정, KMO 표본적합도)
5. 요인 수 결정 (고유값, Scree Plot)
6. 요인분석 실시 (최대우도법 + 사각회전)
7. 요인적재량 해석 및 요인 명명
8. 공통성(communality) 확인
9. 요인점수 산출 및 저장
10. 심화 분석 - 요인점수와 실제 만족도 응답 간 관계 시각화
11. 결론 및 레포트 요약

## 1. 데이터셋 소개 및 분석 목적

### 1.1 데이터 출처

본 분석에 사용한 데이터셋은 Kaggle에 공개된 **Airline Passenger Satisfaction** 데이터셋이다.

- 출처: https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction
- 내용: 어느 항공사가 자사 승객을 대상으로 실시한 만족도 설문조사 결과
- 원본 데이터는 train/test로 분할되어 배포되며, 본 과제에서는 그중 `test.csv` 파일(약 2만
  6천 건)을 사용한다.

### 1.2 데이터셋 구성

데이터셋은 승객 1인당 1행으로 구성되어 있으며, 크게 세 종류의 변수로 나눌 수 있다.

| 구분 | 변수 | 설명 |
|---|---|---|
| 승객 특성 | Gender, Customer Type, Age, Type of Travel, Class | 성별, 고객 유형(충성/비충성 고객), 연령, 여행 목적(출장/개인), 좌석 등급 |
| 비행 정보 | Flight Distance, Departure Delay in Minutes, Arrival Delay in Minutes | 비행 거리, 출발/도착 지연 시간(분) |
| 서비스 만족도 문항 (Likert, 0~5점) | Inflight wifi service, Departure/Arrival time convenient, Ease of Online booking, Gate location, Food and drink, Online boarding, Seat comfort, Inflight entertainment, On-board service, Leg room service, Baggage handling, Checkin service, Inflight service, Cleanliness | 기내/지상 서비스 각 항목에 대한 만족도 평점 (0: 해당없음/무응답, 1~5점: 만족도) |
| 종속/결과 변수 | satisfaction | 최종 만족 여부 (satisfied / neutral or dissatisfied) |

### 1.3 분석 목적

위 표의 세 번째 범주, 즉 **14개의 서비스 만족도 평가 문항**이 본 요인분석의 분석 대상이다.
이 14개 문항은 와이파이, 좌석, 기내식, 승무원 서비스 등 서로 다른 영역을 각각 하나의
문항으로 측정하고 있지만, 실제로는 소수의 더 큰 잠재적 차원(예: "기내 쾌적성", "서비스
응대 품질", "예약·탑승 편의성" 등)으로 묶일 가능성이 있다.

따라서 본 분석의 목적은 다음과 같다.

1. 14개의 개별 서비스 만족도 문항이 몇 개의 공통된 잠재요인으로 축약될 수 있는지 확인한다.
2. 각 요인을 구성하는 문항들을 바탕으로 요인의 의미를 해석하고 이름을 부여한다.
3. 산출된 요인점수가 실제 승객의 최종 만족 여부(satisfaction)와 어떤 관계를 보이는지
   시각적으로 살펴봄으로써 요인분석 결과의 타당성을 간접적으로 확인한다.

나이, 비행 거리, 지연 시간, 성별 등은 서비스에 대한 '태도/평가' 문항이 아니라 승객의
인구통계학적·상황적 특성이므로 요인분석 대상에서 제외한다.

## 2. 환경 설정 및 데이터 불러오기

In [ ]:
# !pip install pandas numpy scipy matplotlib seaborn factor_analyzer scikit-learn

# 데이터 분석
import numpy as np
import pandas as pd
from pathlib import Path

# 요인분석
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
from factor_analyzer import FactorAnalyzer

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 메시지 정리 (요인분석 적합 과정에서 발생하는 수렴 관련 FutureWarning 숨김)
import warnings
warnings.filterwarnings(action='ignore', category=FutureWarning)

# 한글 출력 설정 (Windows 기준 맑은고딕 폰트 사용)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

### 참고: scikit-learn 버전 호환성

`factor_analyzer` 패키지(0.5.1 기준)는 내부적으로 scikit-learn의 `check_array` 함수를
`force_all_finite` 인자와 함께 호출하는데, 이 인자는 최신 scikit-learn(1.6 이상)에서
제거되어 `TypeError: check_array() got an unexpected keyword argument 'force_all_finite'`
오류가 발생할 수 있다. 해당 오류가 발생하면 아래 명령으로 scikit-learn 버전을 낮추고
커널을 재시작하면 해결됩니다.

```
pip install "scikit-learn<1.6"
```

### CSV 경로 설정

In [ ]:
BASE_DIR = Path.cwd()                      # 현재 노트북(practice 폴더) 위치
SOURCE_DIR = BASE_DIR.parent / 'source'    # data-analysis-practice/source 폴더

print('노트북 위치   :', BASE_DIR)
print('데이터 폴더   :', SOURCE_DIR)
print('test.csv 존재 여부:', (SOURCE_DIR / 'test.csv').exists())

In [ ]:
data = pd.read_csv(SOURCE_DIR / 'test.csv')

print('원본 데이터 크기(행, 열):', data.shape)
data.head()

## 3. 데이터 전처리

### 3.1 분석 대상 변수 선택

요인분석은 서비스 만족도를 측정하는 14개의 Likert 문항만을 대상으로 한다. `Unnamed: 0`,
`id`는 단순 식별자이므로 제외하고, 인구통계·비행정보 변수(Age, Flight Distance, Delay 등)와
결과 변수(satisfaction)도 요인분석 자체에는 포함하지 않는다(다만 satisfaction은 10장의
심화 분석에서 다시 활용한다).

In [ ]:
likert_cols = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
    'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness'
]

df = data[likert_cols]
print('요인분석 대상 변수 개수:', len(likert_cols))
df.describe().T

### 3.2 결측치 및 이상값(0점) 처리

기술통계를 보면 각 문항의 응답 범위는 이론적으로 1~5점이어야 하는데 최솟값이 0으로
나타나는 문항들이 있다. 이는 Kaggle 데이터셋 설명 및 값의 분포를 볼 때 해당 서비스를
이용하지 않았거나 응답하지 않은 '결측에 가까운 값'으로 보는 것이 합리적이다(예: 공항
좌석 위치(Gate location)처럼 거의 모든 승객이 경험하는 항목은 0이 전혀 없는 반면,
와이파이·온라인 예약처럼 이용하지 않았을 수 있는 항목에서만 0이 나타난다).

따라서 0점을 실제 결측치(NaN)로 바꾼 뒤, 14개 문항 중 하나라도 결측인 행은 분석에서
제외한다.

In [ ]:
# 문항별 0점 비율 확인
zero_ratio = (df == 0).mean().sort_values(ascending=False) * 100
print('문항별 0점(결측 추정) 비율(%)')
print(zero_ratio.round(2))

In [ ]:
# 0점을 결측치로 변환 후, 결측이 있는 행 제거
df_clean = df.replace(0, np.nan)
before_n = len(df_clean)
df_clean = df_clean.dropna().reset_index(drop=True)
after_n = len(df_clean)

print(f'전처리 전 표본 수 : {before_n:,}')
print(f'전처리 후 표본 수 : {after_n:,}')
print(f'제외된 표본 비율  : {(before_n - after_n) / before_n * 100:.2f}%')

결측치 제거 후에도 약 23,800건 이상의 표본이 남아, 요인분석에 일반적으로 권장되는
표본 크기(문항 수 대비 최소 5~10배, 즉 14개 문항 기준 약 70~140건)를 충분히 상회한다.
따라서 결측 행을 제거하는 방식으로 처리해도 분석의 통계적 검정력에는 문제가 없다.

In [ ]:
df_clean.shape

In [ ]:
df_clean.describe().T.round(2)

## 4. 탐색적 데이터 분석 - 문항 간 상관관계

요인분석은 변수들 간의 상관관계 구조를 바탕으로 잠재요인을 추출하는 기법이므로,
본격적인 분석에 앞서 14개 문항 간의 상관행렬을 시각적으로 확인한다. 상관계수가
전반적으로 지나치게 낮으면(거의 0에 가까우면) 요인분석을 적용할 근거가 약해진다.

In [ ]:
plt.figure(figsize=(11, 9))
corr = df_clean.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('14개 서비스 만족도 문항 간 상관행렬', fontsize=14)
plt.tight_layout()
plt.show()

상관행렬을 보면 일부 문항 간에는 상관계수가 0.5를 넘는 뚜렷한 상관관계가 관찰되는
반면(예: Inflight entertainment-Cleanliness, Baggage handling-Inflight service 등),
일부 문항(예: Checkin service)은 다른 문항들과의 상관관계가 전반적으로 낮다. 이는
문항들이 몇 개의 군집(잠재요인)으로 묶일 가능성을 시사하며, 동시에 일부 문항은 공통
요인으로 잘 설명되지 않을 수 있음을 미리 예상하게 한다.

## 5. 요인분석 적합성 검정

요인분석을 실시하기 전에, 현재 데이터가 요인분석에 적합한 상관구조를 가지고 있는지
확인하는 두 가지 사전 검정을 실시한다.

- **Bartlett의 구형성 검정(Bartlett's test of sphericity)**: 변수들 간의 상관행렬이
  단위행렬(모든 상관계수가 0)인지 검정한다. 귀무가설(상관행렬=단위행렬)이 기각되어야
  (p < .05) 요인분석을 실시할 근거가 있다고 본다.
- **KMO 표본적합도(Kaiser-Meyer-Olkin measure of sampling adequacy)**: 변수들이
  공통요인으로 설명될 수 있는 정도를 0~1 사이의 값으로 나타낸다. 일반적으로 0.6 이상이면
  요인분석에 적합하다고 판단하며, 0.8 이상이면 '우수(meritorious)'로 해석한다.

In [ ]:
chi_square_value, p_value = calculate_bartlett_sphericity(df_clean)

print('Bartlett 구형성 검정')
print(f'  Chi-square = {chi_square_value:,.2f}')
print(f'  p-value    = {p_value:.4f}')

Bartlett 검정 결과 p-value가 .05보다 훨씬 작게 나타나(사실상 0에 가까움), 귀무가설
(상관행렬이 단위행렬이다)을 기각한다. 즉 문항들 사이에 유의미한 상관관계가 존재하므로
요인분석을 실시할 통계적 근거가 확보되었다.

In [ ]:
kmo_per_item, kmo_total = calculate_kmo(df_clean)

print(f'KMO 전체 표본적합도: {kmo_total:.3f}')
print()
print('문항별 KMO 값')
print(pd.Series(kmo_per_item, index=df_clean.columns).sort_values().round(3))

전체 KMO 값이 약 0.79로 나타나 통상적인 기준(0.6 이상)을 넉넉히 상회하며, '양호
(middling~meritorious)' 수준에 해당한다. 문항별 KMO 값도 대부분 0.7 이상으로 안정적인
수준이어서, 특정 문항을 제거해야 할 만큼 부적합한 변수는 눈에 띄지 않는다. 두 검정 결과를
종합하면 현재 데이터셋은 요인분석을 적용하기에 적합하다고 판단할 수 있다.

## 6. 요인 수 결정

요인의 개수는 보통 다음 두 가지 기준을 함께 고려하여 결정한다.

1. **Kaiser 기준**: 고유값(eigenvalue)이 1보다 큰 요인만 채택한다.
2. **Scree Plot(스크리 도표)**: 고유값을 내림차순으로 그래프에 그렸을 때, 기울기가
   완만해지기 시작하는 지점(elbow) 직전까지의 요인 수를 채택한다.

먼저 회전을 적용하지 않은 상태로 요인분석을 적합시켜 고유값을 계산한다.

In [ ]:
fa_initial = FactorAnalyzer(rotation=None)
fa_initial.fit(df_clean)

ev, v = fa_initial.get_eigenvalues()

print('고유값(eigenvalue) 목록 (내림차순)')
for i, val in enumerate(ev, start=1):
    marker = '  <- 채택 (고유값 > 1)' if val > 1 else ''
    print(f'  요인 {i:2d}: {val:.3f}{marker}')

In [ ]:
num_factors = int(sum(ev > 1))
print(f'Kaiser 기준(고유값 > 1)에 따라 채택된 요인 수: {num_factors}개')

In [ ]:
plt.figure(figsize=(8, 5))
x = range(1, len(ev) + 1)
plt.plot(x, ev, marker='o', linestyle='-', color='steelblue', label='고유값')
plt.axhline(y=1, color='red', linestyle='--', linewidth=1, label='기준선 (eigenvalue = 1)')
plt.axvline(x=num_factors, color='gray', linestyle=':', linewidth=1)

plt.title('Scree Plot - 요인별 고유값', fontsize=14)
plt.xlabel('요인 번호')
plt.ylabel('고유값(Eigenvalue)')
plt.xticks(x)
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

Scree Plot을 보면 1~4번째 요인까지는 고유값이 비교적 완만하게 감소하다가, 4번째 요인
이후부터 기울기가 눈에 띄게 완만해지며 1 이하로 떨어진다(elbow 지점). 이는 Kaiser 기준
(고유값 > 1)으로 산출된 4개 요인이라는 결과와 일치한다. 따라서 이후 분석에서는
**4개의 요인**을 추출하기로 결정한다.

## 7. 요인분석 실시

요인추출 방법으로는 수업에서 다룬 것과 동일하게 **최대우도법(Maximum Likelihood, ML)**을
사용하고, 요인 간 상관을 허용하는 **사각회전(oblimin rotation)**을 적용한다. 서비스
만족도를 구성하는 잠재적 차원들(예: 기내 편의성, 응대 서비스 품질 등)은 서로 독립적이라기
보다는 어느 정도 상관되어 있을 가능성이 높으므로(만족도가 전반적으로 높은 승객은 여러
요인에서 동시에 높은 점수를 줄 가능성), 요인 간 상관을 가정하지 않는 직각회전(varimax)보다
사각회전이 더 현실적인 가정이라고 판단했다.

In [ ]:
FA = FactorAnalyzer(
    n_factors=num_factors,
    method='ml',
    rotation='oblimin',
    rotation_kwargs={'max_iter': 100}
)
FA.fit(df_clean)

print('요인분석 적합 완료')
print(f'추출 요인 수 : {num_factors}')
print(f'추출 방법   : 최대우도법(ML)')
print(f'회전 방법   : 사각회전(oblimin)')

### 7.1 요인적재량(Factor Loadings)

In [ ]:
loadings = pd.DataFrame(
    FA.loadings_,
    index=df_clean.columns,
    columns=[f'Factor{i+1}' for i in range(num_factors)]
)
loadings.round(3)

### 7.2 해석을 돕기 위한 요인적재량 정리

절댓값 기준 0.3 미만의 적재량은 해당 문항이 그 요인을 유의미하게 구성하지 않는다고 보고
공백으로 처리하여, 각 요인에 실질적으로 기여하는 문항만 한눈에 볼 수 있도록 정리한다.

In [ ]:
loadings_blank = loadings.copy()
loadings_blank = loadings_blank.where(loadings_blank.abs() > 0.3, '')
loadings_blank

### 7.3 요인적재량 히트맵 시각화

In [ ]:
plt.figure(figsize=(7, 8))
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5, cbar_kws={'label': '요인적재량'})
plt.title('요인적재량 히트맵 (ML 추출, oblimin 회전)', fontsize=14)
plt.xlabel('요인')
plt.ylabel('문항')
plt.tight_layout()
plt.show()

히트맵에서 색이 진하게 나타나는 셀일수록(파란색: 양의 적재, 빨간색: 음의 적재) 해당
문항이 그 요인을 강하게 설명한다는 의미이다. 문항들이 네 개의 요인에 비교적 뚜렷하게
나뉘어 적재되는 것을 확인할 수 있다.

### 7.4 요인 해석 및 명명

각 요인에 높은 적재량(절댓값 0.3 이상)을 보이는 문항들을 근거로 요인의 의미를 다음과
같이 해석하고 이름을 부여한다.

- **Factor1 (기내 편의성)**: Food and drink, Seat comfort, Inflight entertainment,
  Cleanliness — 기내에서 체감하는 물리적 쾌적함 및 편의 시설과 관련된 문항들
- **Factor2 (승무원/지상 서비스 품질)**: On-board service, Leg room service,
  Baggage handling, Inflight service(, Checkin service) — 승무원 응대 및 수하물 처리 등
  '사람에 의해 제공되는 서비스'와 관련된 문항들
- **Factor3 (예약 및 접근 편의성)**: Inflight wifi service, Departure/Arrival time
  convenient, Ease of Online booking, Gate location — 비행 전후 예약·접근과 관련된 편의성
  문항들
- **Factor4 (온라인 탑승수속)**: Online boarding — 온라인 탑승수속 한 문항이 독립적으로
  강하게 적재되는 요인

Factor4는 다른 요인과 달리 사실상 'Online boarding' 한 문항이 단독으로 구성하는 요인에
가깝다(와이파이 문항이 다소 교차 적재되기는 하나 절대값이 크지 않다). 이는 통계적으로는
독립된 요인으로 분리되었지만, 개념적으로는 단일 문항 요인이라는 한계가 있어 10장의
결론에서 다시 논의한다.

### 7.5 요인명으로 컬럼명 변경

In [ ]:
loadings_named = loadings.copy()
loadings_named.columns = ['기내편의성', '서비스품질', '예약접근편의성', '온라인탑승']
loadings_named.round(3)

## 8. 공통성(Communality) 확인

공통성은 각 문항의 분산 중 추출된 요인들에 의해 설명되는 비율을 의미한다. 일반적으로
공통성이 0.4(혹은 연구 목적에 따라 0.3) 미만이면 해당 문항이 추출된 요인 구조로 잘
설명되지 않는 문항으로 간주하여 제거를 고려하기도 한다.

In [ ]:
communality = pd.DataFrame(
    FA.get_communalities(),
    index=df_clean.columns,
    columns=['공통성']
).sort_values('공통성', ascending=False)

communality.round(3)

In [ ]:
plt.figure(figsize=(8, 5))
colors = ['steelblue' if c >= 0.4 else 'indianred' for c in communality['공통성']]
plt.barh(communality.index, communality['공통성'], color=colors)
plt.axvline(x=0.4, color='black', linestyle='--', linewidth=1, label='기준선 (0.4)')
plt.title('문항별 공통성(Communality)', fontsize=14)
plt.xlabel('공통성')
plt.legend()
plt.tight_layout()
plt.show()

대부분의 문항은 공통성이 0.4 이상으로, 추출된 4개 요인 구조로 비교적 잘 설명된다.
다만 **Checkin service**는 공통성이 뚜렷하게 낮게 나타나 4개 요인 중 어디에도 강하게
속하지 않는, 상대적으로 설명력이 낮은 문항으로 확인된다. 이는 7.4절에서 Checkin service를
Factor2에 약하게만 포함시킨 것과 일치하는 결과이며, 추후 설문을 재설계한다면 이 문항의
표현을 다듬거나 다른 문항으로 보완할 필요가 있음을 시사한다.

## 9. 요인점수 산출 및 저장

요인점수(factor score)는 각 응답자가 추출된 각 요인에 대해 가지는 상대적인 수준을
하나의 점수로 환산한 값이다. 이를 산출해두면, 이후 집단 간 비교나 회귀분석 등 후속
분석에서 14개의 개별 문항 대신 4개의 요약된 요인점수를 변수로 사용할 수 있다.

In [ ]:
factor_scores = FA.transform(df_clean)

fs_df = pd.DataFrame(
    factor_scores,
    columns=['기내편의성', '서비스품질', '예약접근편의성', '온라인탑승']
)
fs_df.head()

In [ ]:
fs_df.describe().round(3)

In [ ]:
# 요인점수를 csv 파일로 저장 (한글 깨짐 방지를 위해 utf-8-sig 인코딩 사용)
fs_df.to_csv('factor_scores.csv', index=False, encoding='utf-8-sig')
print('factor_scores.csv 저장 완료 (행:', fs_df.shape[0], ', 열:', fs_df.shape[1], ')')

## 10. 심화 분석 - 요인점수와 실제 만족도 응답 간 관계

요인분석 자체는 satisfaction(최종 만족 여부) 변수를 사용하지 않았지만, 산출된 요인점수가
실제로 의미 있는 정보를 담고 있다면 satisfaction과 체계적인 관련성을 보여야 한다. 이를
간접적인 타당성 확인(construct validity check)으로 활용하기 위해, 전처리 과정에서
함께 남아 있는 satisfaction 값을 요인점수에 다시 결합하여 비교한다.

In [ ]:
# 0값을 결측으로 바꾸고 dropna할 때 사용했던 것과 동일한 인덱스를 satisfaction에도 적용
satisfaction = data[likert_cols].replace(0, np.nan)
satisfaction = satisfaction.join(data['satisfaction']).dropna(subset=likert_cols).reset_index(drop=True)

fs_with_target = fs_df.copy()
fs_with_target['satisfaction'] = satisfaction['satisfaction'].values
fs_with_target.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
factor_names = ['기내편의성', '서비스품질', '예약접근편의성', '온라인탑승']

for ax, name in zip(axes.flat, factor_names):
    sns.boxplot(data=fs_with_target, x='satisfaction', y=name, hue='satisfaction',
                palette='Set2', legend=False, ax=ax)
    ax.set_title(f'{name} 요인점수 분포')
    ax.set_xlabel('')
    ax.set_ylabel('요인점수')

plt.suptitle('만족 여부(satisfaction)에 따른 요인점수 분포 비교', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fs_with_target.groupby('satisfaction')[factor_names].mean().round(3)

In [ ]:
# 두 집단(satisfied vs neutral or dissatisfied) 간 요인점수 평균 차이
group_mean = fs_with_target.groupby('satisfaction')[factor_names].mean()
gap = (group_mean.loc['satisfied'] - group_mean.loc['neutral or dissatisfied']).sort_values(ascending=False)

print('요인별 평균 차이 (satisfied - neutral or dissatisfied), 큰 순')
print(gap.round(3))

박스플롯과 두 집단 간 평균 차이를 함께 보면, 4개 요인 모두 만족(satisfied) 집단에서
점수가 더 높게 나타나 전반적으로 요인점수가 실제 만족 여부와 일관된 방향으로 움직인다는
것을 확인할 수 있다. 다만 요인 간 차이의 크기에는 뚜렷한 순서가 있다.

- **온라인탑승**과 **기내편의성**은 두 집단 간 평균 차이가 가장 크게 나타나, 만족 여부를
  가장 잘 구분해 주는 요인으로 확인된다. 특히 '온라인탑승'은 7.4절에서 사실상 단일 문항
  요인이라는 구조적 한계를 지적했음에도 불구하고, 실제 만족 여부와의 관련성은 네 요인 중
  가장 강하게 나타났다. 이는 온라인 탑승수속이라는 단일 경험이 승객의 전반적인 만족도
  평가에 실질적으로 큰 영향을 미치는 요소임을 시사한다.
- 반대로 **예약접근편의성** 요인은 두 집단 간 평균 차이가 네 요인 중 가장 작게 나타난다.
  즉 와이파이, 온라인 예약, 게이트 위치 등 예약·접근 관련 편의성은 다른 요인들에 비해
  최종 만족 여부를 가르는 변별력이 상대적으로 약하다고 해석할 수 있다.

## 11. 결론 및 레포트 요약

### 11.1 분석 요약

본 레포트에서는 Kaggle의 Airline Passenger Satisfaction 데이터셋에 포함된 14개의 서비스
만족도 Likert 문항을 대상으로 탐색적 요인분석(EFA)을 실시했다. Bartlett 구형성 검정과
KMO 표본적합도 검정을 통해 데이터가 요인분석에 적합함을 확인했으며, Kaiser 기준과
Scree Plot을 근거로 4개의 요인을 추출했다. 최대우도법(ML)과 사각회전(oblimin)을 적용한
결과, 14개 문항은 다음 네 가지 잠재요인으로 요약되었다.

1. **기내편의성** - 기내식, 좌석 편안함, 기내 엔터테인먼트, 청결도
2. **서비스품질** - 승무원 서비스, 레그룸, 수하물 처리, 기내 서비스
3. **예약접근편의성** - 와이파이, 출도착 시간 편의성, 온라인 예약, 게이트 위치
4. **온라인탑승** - 온라인 탑승수속(단일 문항 성격)

공통성 분석 결과 대부분의 문항은 추출된 요인 구조로 비교적 잘 설명되었으나, Checkin
service 문항은 공통성이 낮아 해석에 유의가 필요했다. 또한 산출된 요인점수를 실제
만족 여부(satisfaction)와 비교한 결과, 4개 요인 모두 만족 집단에서 점수가 유의미하게
높게 나타나 분석 결과의 타당성을 뒷받침했다. 그중에서도 온라인탑승과 기내편의성 요인은
만족 여부에 따른 점수 차이가 가장 크게 나타나 변별력이 높은 요인으로 확인되었고,
예약접근편의성 요인은 상대적으로 변별력이 낮았다.

### 11.2 분석의 한계

- 데이터셋에 포함된 0점 응답을 결측으로 간주하고 제거하는 방식으로 처리했는데, 이것이
  실제 설문 설계 의도(진짜 0점인지, 무응답인지)와 다를 가능성을 완전히 배제할 수 없다.
- 'Online boarding' 문항이 사실상 단독으로 하나의 요인을 구성하여, 해당 요인은 이론적
  차원이라기보다 측정 방식(문항 구성)에 따른 결과일 가능성이 있다.
- 본 분석은 탐색적 요인분석(EFA)이므로, 도출된 4요인 구조가 다른 표본에서도 동일하게
  재현되는지는 확인적 요인분석(CFA) 등 추가 검증이 필요하다.

### 11.3 의의

항공사 입장에서는 14개의 개별 서비스 항목을 모두 따로 관리하기보다, 본 분석에서 도출된
4개의 상위 차원(기내편의성, 서비스품질, 예약접근편의성, 온라인탑승) 단위로 서비스 개선
우선순위를 설정하는 것이 더 효율적인 전략이 될 수 있다. 특히 만족 여부와의 관련성이 가장
크게 나타난 온라인탑승과 기내편의성 요인에 자원을 우선 배분하는 것이 승객 만족도 제고에
효과적일 것으로 판단된다.